# LLM-as-Judge Evaluation: Loan Rejection Letters

**Scenario:** Nordbank, a German retail bank, drafts consumer loan rejection letters
with an LLM. A human loan officer reviews and signs every letter.

**What this notebook does**

1. Loads five test cases built from the decision-record format defined in `evaluation_design.md`.
2. Runs **deterministic checks** first (reason fidelity, mandatory elements, word count).
   These are cheap, reproducible, and gate everything else.
3. Runs an **LLM-as-judge** on tone, using the judge prompt from Step 4.
4. Collects score, latency, token usage and estimated cost per case.
5. Writes `evaluation_results.json` and prints a summary.

**Design note:** the judge scores tone *only*. Accuracy and compliance elements are
checked by code. Using a judge for a binary string check would be a cost and
reliability regression, not an upgrade.

## Step 7 — Setup & Environment

Install dependencies (run once), then load your API key from `.env`.

`.env` must contain:

```
OPENAI_API_KEY=sk-...
```

and `.env` must be listed in `.gitignore` before you commit anything.

In [ ]:
# Run once. Comment out afterwards.
# %pip install openai python-dotenv

In [ ]:
import os, json, time, re
from datetime import datetime, timezone

from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()

api_key = os.getenv("OPENAI_API_KEY")
if not api_key:
    raise RuntimeError(
        "OPENAI_API_KEY not found. Create a .env file with OPENAI_API_KEY=your_key_here"
    )

client = OpenAI(api_key=api_key)

JUDGE_MODEL = "gpt-4o-mini"
TEMPERATURE = 0          # judges should be as reproducible as the API allows

# Pricing in USD per 1M tokens. VERIFY against current OpenAI pricing before
# quoting these numbers to anyone. gpt-4o-mini is cheap, but it is not free.
PRICE_INPUT_PER_1M = 0.15
PRICE_OUTPUT_PER_1M = 0.60

print("Client ready. Judge model:", JUDGE_MODEL)

## Step 8 — Implement the Judge

This is the judge prompt from Step 4, verbatim, with the letter injected at call time.

Two things worth noticing in the code below:

- `response_format={"type": "json_object"}` removes most JSON parsing failures at the source.
- The parse is still wrapped in `try/except`. A judge that returns malformed output is a
  **result**, not a crash: we record it as a failed evaluation rather than losing the run.

In [ ]:
JUDGE_SYSTEM_PROMPT = """
You are evaluating a draft loan rejection letter written by a language model for a
German retail bank. A human loan officer will review and sign the letter before it is
sent. Your job is to score the draft, not to rewrite it.

TASK DESCRIPTION
The model received a structured decision record and was asked to write the
applicant-facing rejection letter. You are assessing TONE ONLY. Factual accuracy and
the presence of mandatory elements are checked separately by automated rules. Do not
score them, and do not lower a score because a required sentence is missing.

EVALUATION CRITERIA

1. Non-condescension. The letter states the finding without lecturing, moralising, or
   implying the applicant is financially irresponsible. Neutral factual statements
   about payment history are correct and must NOT be penalised. Unsolicited advice,
   and language positioning the bank as an instructor, are failures.

2. No false hope. The letter does not imply the decision may be reversed, that
   reapplying soon is likely to succeed, or that the outcome is provisional. Note
   carefully: informing the applicant of their right to request a manual review is
   REQUIRED and is NOT false hope. False hope is a prediction about the outcome, not
   a statement of a procedural right.

3. Plain language. A reader without financial training understands the letter on one
   reading. Failures include unexplained jargon, sentences over roughly 25 words,
   nested subordinate clauses, and bureaucratic passive constructions that hide who
   decided what.

REASONING STEPS
Step 1: Read the letter once and identify the sentence communicating the decline. Quote it.
Step 2: Assess criterion 1. Look for evaluative language about the applicant as a
        person, as distinct from factual language about the account.
Step 3: Assess criterion 2. Separate procedural statements from predictive statements.
        Only predictive statements can create false hope.
Step 4: Assess criterion 3. Identify the least readable sentence and judge whether a
        lay reader would follow it.
Step 5: Assign the overall score. Take the LOWEST of the three criteria, not the
        average. A letter that is warm and readable but condescending is not passable.

SCORE ANCHORS
5 = Sendable as drafted. Neutral, clear, no implied promises.
4 = Sendable after a trivial edit of one phrase.
3 = Usable structure, but one criterion clearly fails; a human must rewrite a sentence.
2 = Two criteria fail, or the tone would plausibly generate a complaint.
1 = The bank would not send this. Openly judgemental, or promises an outcome it
    cannot deliver.

OUTPUT FORMAT
Return only valid JSON, no preamble and no code fences:

{
  "decline_sentence": "the sentence you quoted in Step 1",
  "score": 1-5,
  "reasoning": "two to four sentences citing specific wording",
  "criteria_met": {
    "non_condescension": true/false,
    "no_false_hope": true/false,
    "plain_language": true/false
  },
  "lowest_criterion": "name of the weakest criterion"
}
""".strip()


def judge_letter(decision_record: str, letter: str, model: str = JUDGE_MODEL) -> dict:
    """Score one draft letter on tone.

    Returns a dict with the judge verdict plus latency, token and cost metrics.
    Never raises on a bad judge response; records the failure instead.
    """
    user_content = (
        f"DECISION RECORD\n{decision_record}\n\n"
        f"DRAFT LETTER\n{letter}\n\n"
        "Score this letter now. Return only the JSON object."
    )

    start = time.perf_counter()
    response = client.chat.completions.create(
        model=model,
        temperature=TEMPERATURE,
        response_format={"type": "json_object"},
        messages=[
            {"role": "system", "content": JUDGE_SYSTEM_PROMPT},
            {"role": "user", "content": user_content},
        ],
    )
    elapsed = time.perf_counter() - start

    raw = response.choices[0].message.content
    try:
        verdict = json.loads(raw)
        parse_ok = True
    except json.JSONDecodeError:
        verdict = {"score": None, "reasoning": "JUDGE RETURNED UNPARSEABLE OUTPUT",
                   "criteria_met": {}, "raw_output": raw}
        parse_ok = False

    usage = response.usage
    cost = (usage.prompt_tokens / 1_000_000) * PRICE_INPUT_PER_1M + \
           (usage.completion_tokens / 1_000_000) * PRICE_OUTPUT_PER_1M

    verdict["_meta"] = {
        "parse_ok": parse_ok,
        "model": model,
        "seconds": round(elapsed, 3),
        "input_tokens": usage.prompt_tokens,
        "output_tokens": usage.completion_tokens,
        "estimated_cost_usd": round(cost, 6),
    }
    return verdict

### Smoke test the judge before spending money on the full run

Two deliberately opposite letters. If the judge cannot separate these, the prompt is
broken and there is no point running the rest.

In [ ]:
SMOKE_RECORD = "primary_reason: adverse payment history on an existing obligation"

SMOKE_BAD = """Dear Mr Braun,

Unfortunately we must decline your application. Frankly, your repayment record speaks
for itself, and we would encourage you to reflect on your spending habits before
approaching a lender again. If you can demonstrate better discipline over the coming
months, we are confident this decision could look very different.

Kind regards, Nordbank"""

SMOKE_GOOD = """Dear Mr Braun,

We have declined your application for a consumer loan. The decision is based on the
payment history recorded on an existing obligation with us.

You may request a manual review of this decision. To do so, contact us in writing at
the address above within four weeks of the date of this letter.

Yours sincerely, Nordbank"""

for label, text in [("BAD", SMOKE_BAD), ("GOOD", SMOKE_GOOD)]:
    v = judge_letter(SMOKE_RECORD, text)
    print(f"{label}: score={v.get('score')}  weakest={v.get('lowest_criterion')}")
    print("   ", v.get("reasoning", "")[:220], "\n")

## Step 9 — Test Dataset

Five cases, one per evaluation prompt from Step 3.

The letters below are **pre-written samples**, not model output. That is a deliberate
choice: pre-written cases let us control exactly which failure each case contains, so
we can tell whether the judge detects a known fault. Generating them live would test
the drafting model and the judge at the same time and confound the two.

Case 5 is the important one. The decision record has an empty `primary_reason`, and the
sample letter invents a plausible reason to fill the gap. Correct behaviour would have
been to flag the missing field and refuse to produce a finished letter.

In [ ]:
TEST_CASES = [
    {
        "id": "TC1_reason_fidelity",
        "eval_prompt": "Prompt 1 — Reason fidelity (single reason)",
        "decision_record": {
            "applicant_name": "Frau Keller",
            "product": "Consumer loan, 12000 EUR, 48 months",
            "decision_date": "2026-08-14",
            "primary_reason": "insufficient documented income for the requested term",
            "secondary_reason": None,
            "bureau_consulted": None,
            "bureau_score_used": False,
        },
        "letter": """Dear Frau Keller,

We have declined your application for a consumer loan of 12,000 EUR over 48 months.
The documented income you provided is not sufficient to support repayment over the
term requested.

You may request a manual review of this decision by writing to us within four weeks.

Yours sincerely, Nordbank""",
        "expected": {
            "required_reason": "income",
            "forbidden_reasons": ["payment history", "existing obligations",
                                  "credit score", "employment duration"],
            "bureau_expected": False,
            "should_be_a_letter": True,
        },
    },
    {
        "id": "TC2_mandatory_elements",
        "eval_prompt": "Prompt 2 — Mandatory element presence",
        "decision_record": {
            "applicant_name": "Herr Vogel",
            "product": "Consumer loan, 8000 EUR, 36 months",
            "decision_date": "2026-08-14",
            "primary_reason": "level of existing credit obligations",
            "secondary_reason": None,
            "bureau_consulted": "SCHUFA",
            "bureau_score_used": True,
        },
        "letter": """Dear Herr Vogel,

We have declined your application for a consumer loan. The decision reflects the level
of your existing credit obligations relative to your income.

In reaching this decision we consulted information held by SCHUFA.

You have the right to request a manual review. Please write to us at the address above
within four weeks of the date of this letter.

Yours sincerely, Nordbank""",
        "expected": {
            "required_reason": "existing credit obligations",
            "forbidden_reasons": ["insufficient income", "employment duration"],
            "bureau_expected": True,
            "should_be_a_letter": True,
        },
    },
    {
        "id": "TC3_tone_under_pressure",
        "eval_prompt": "Prompt 3 — Tone under pressure (judged case)",
        "decision_record": {
            "applicant_name": "Herr Braun",
            "product": "Consumer loan, 15000 EUR, 60 months",
            "decision_date": "2026-08-14",
            "primary_reason": "adverse payment history on an existing obligation",
            "secondary_reason": None,
            "bureau_consulted": "SCHUFA",
            "bureau_score_used": True,
            "note": "customer of eleven years",
        },
        "letter": """Dear Herr Braun,

After eleven years we know you well, which makes this letter harder to write. We have
to decline your loan application because of missed payments on an existing obligation.
We would gently encourage you to keep a closer eye on due dates going forward, and we
are optimistic that a future application will have a much better outcome.

We consulted SCHUFA in reaching this decision. You may ask for a manual review.

Warmly, Nordbank""",
        "expected": {
            "required_reason": "payment history",
            "forbidden_reasons": ["insufficient income"],
            "bureau_expected": True,
            "should_be_a_letter": True,
            "known_fault": "condescension plus false hope",
        },
    },
    {
        "id": "TC4_conditional_no_bureau",
        "eval_prompt": "Prompt 4 — Conditional logic (bureau NOT used)",
        "decision_record": {
            "applicant_name": "Frau Adler",
            "product": "Consumer loan, 5000 EUR, 24 months",
            "decision_date": "2026-08-14",
            "primary_reason": "level of existing credit obligations",
            "secondary_reason": None,
            "bureau_consulted": None,
            "bureau_score_used": False,
        },
        "letter": """Dear Frau Adler,

We have declined your application for a consumer loan of 5,000 EUR. The decision
reflects the level of your existing credit obligations.

Our assessment took account of your SCHUFA score.

You may request a manual review within four weeks.

Yours sincerely, Nordbank""",
        "expected": {
            "required_reason": "existing credit obligations",
            "forbidden_reasons": ["insufficient income"],
            "bureau_expected": False,
            "should_be_a_letter": True,
            "known_fault": "asserts a bureau contribution that did not occur",
        },
    },
    {
        "id": "TC5_incomplete_record",
        "eval_prompt": "Prompt 5 — Adversarial, incomplete record",
        "decision_record": {
            "applicant_name": "Herr Sommer",
            "product": "Consumer loan, 20000 EUR, 60 months",
            "decision_date": "2026-08-14",
            "primary_reason": "",
            "secondary_reason": None,
            "bureau_consulted": "SCHUFA",
            "bureau_score_used": True,
        },
        "letter": """Dear Herr Sommer,

We regret to inform you that your application for a consumer loan has been declined.
The requested amount exceeds what your current income profile can support.

You may request a manual review within four weeks.

Yours sincerely, Nordbank""",
        "expected": {
            "required_reason": None,          # there is no reason to state
            "forbidden_reasons": ["income", "payment history",
                                  "existing credit obligations", "employment duration"],
            "bureau_expected": True,
            "should_be_a_letter": False,      # correct behaviour is to FLAG, not draft
            "known_fault": "invented a decline reason for an empty field",
        },
    },
]

print(f"{len(TEST_CASES)} test cases loaded.")

### Deterministic checks

These run before the judge. They are free, instant, and give the same answer every time.
Anything checkable this way should never be sent to a model.

In [ ]:
BUREAU_NAMES = ["schufa", "creditreform", "crif", "bureau"]

REASON_KEYWORDS = {
    "income": ["income", "einkommen", "earnings"],
    "payment history": ["payment history", "missed payment", "arrears", "repayment record"],
    "existing credit obligations": ["existing credit", "existing obligation",
                                    "current obligations", "level of your existing"],
    "employment duration": ["length of employment", "employment duration"],
    "credit score": ["credit score", "score"],
    "insufficient income": ["insufficient income", "income you provided is not sufficient",
                            "income profile"],
}


def mentions(text: str, keywords) -> bool:
    low = text.lower()
    return any(k.lower() in low for k in keywords)


def deterministic_checks(case: dict) -> dict:
    letter = case["letter"]
    exp = case["expected"]
    low = letter.lower()
    checks = {}

    # 1. Required reason present
    if exp["required_reason"]:
        kws = REASON_KEYWORDS.get(exp["required_reason"], [exp["required_reason"]])
        checks["required_reason_present"] = mentions(letter, kws)
    else:
        checks["required_reason_present"] = None   # not applicable

    # 2. No forbidden reason present
    leaked = [r for r in exp["forbidden_reasons"]
              if mentions(letter, REASON_KEYWORDS.get(r, [r]))]
    checks["no_forbidden_reason"] = (len(leaked) == 0)
    checks["leaked_reasons"] = leaked

    # 3. Bureau reference matches the record
    bureau_mentioned = any(b in low for b in BUREAU_NAMES)
    checks["bureau_reference_correct"] = (bureau_mentioned == exp["bureau_expected"])
    checks["bureau_mentioned"] = bureau_mentioned

    # 4. Manual review right stated
    checks["manual_review_stated"] = ("manual review" in low or "review of this decision" in low)

    # 5. Word count under 300
    wc = len(re.findall(r"\b\w+\b", letter))
    checks["word_count"] = wc
    checks["under_300_words"] = wc < 300

    booleans = [v for k, v in checks.items()
                if isinstance(v, bool) and k != "bureau_mentioned"]
    checks["gate_passed"] = all(booleans)
    return checks


for case in TEST_CASES:
    r = deterministic_checks(case)
    print(f"{case['id']:<26} gate_passed={str(r['gate_passed']):<6} "
          f"words={r['word_count']:<4} leaked={r['leaked_reasons']}")

## Step 10 — Run the Evaluation & Collect Metrics

The judge runs on every case so that we can compare its tone scores against the
deterministic verdicts. In production the gate would come first and only passing drafts
would reach a judge; here we want the full picture for the report.

In [ ]:
def record_to_text(rec: dict) -> str:
    return "\n".join(f"  {k}: {v}" for k, v in rec.items())


results = []
run_started = datetime.now(timezone.utc).isoformat()
t0 = time.perf_counter()

for case in TEST_CASES:
    det = deterministic_checks(case)
    verdict = judge_letter(record_to_text(case["decision_record"]), case["letter"])

    results.append({
        "id": case["id"],
        "eval_prompt": case["eval_prompt"],
        "known_fault": case["expected"].get("known_fault"),
        "deterministic": det,
        "judge": verdict,
    })
    print(f"{case['id']:<26} judge_score={verdict.get('score')}  "
          f"{verdict['_meta']['seconds']}s  "
          f"${verdict['_meta']['estimated_cost_usd']:.6f}")

total_seconds = time.perf_counter() - t0
print(f"\nTotal wall time: {total_seconds:.2f}s")

In [ ]:
scores = [r["judge"]["score"] for r in results
          if isinstance(r["judge"].get("score"), (int, float))]

aggregate = {
    "run_started_utc": run_started,
    "judge_model": JUDGE_MODEL,
    "temperature": TEMPERATURE,
    "n_cases": len(results),
    "n_judged_successfully": len(scores),
    "mean_score": round(sum(scores) / len(scores), 2) if scores else None,
    "min_score": min(scores) if scores else None,
    "max_score": max(scores) if scores else None,
    "gate_pass_rate": round(
        sum(r["deterministic"]["gate_passed"] for r in results) / len(results), 2),
    "total_seconds": round(total_seconds, 2),
    "mean_seconds_per_eval": round(total_seconds / len(results), 2),
    "total_input_tokens": sum(r["judge"]["_meta"]["input_tokens"] for r in results),
    "total_output_tokens": sum(r["judge"]["_meta"]["output_tokens"] for r in results),
    "total_estimated_cost_usd": round(
        sum(r["judge"]["_meta"]["estimated_cost_usd"] for r in results), 6),
}

payload = {"aggregate": aggregate, "results": results}

with open("evaluation_results.json", "w", encoding="utf-8") as f:
    json.dump(payload, f, indent=2, ensure_ascii=False)

print(json.dumps(aggregate, indent=2))
print("\nWritten to evaluation_results.json")

## Step 11 — Analyse Results

The question that matters is not "what was the average score". It is **did the judge
catch the faults we planted**. Cases 3, 4 and 5 each contain a known defect. Case 3's
fault is a tone fault, so the judge should catch it. Cases 4 and 5 are factual and
compliance faults, which the judge was explicitly told to ignore, so a *high* judge
score there is correct behaviour, not a miss.

In [ ]:
print("=" * 78)
print("PER-CASE RESULTS")
print("=" * 78)

for r in results:
    j, d = r["judge"], r["deterministic"]
    print(f"\n{r['id']}  ({r['eval_prompt']})")
    print(f"  planted fault : {r['known_fault'] or 'none (clean case)'}")
    print(f"  gate passed   : {d['gate_passed']}  "
          f"(reason={d['required_reason_present']}, bureau={d['bureau_reference_correct']}, "
          f"review={d['manual_review_stated']}, words={d['word_count']})")
    if d["leaked_reasons"]:
        print(f"  LEAKED REASON : {d['leaked_reasons']}")
    print(f"  judge score   : {j.get('score')}  (weakest: {j.get('lowest_criterion')})")
    print(f"  criteria      : {j.get('criteria_met')}")
    print(f"  reasoning     : {str(j.get('reasoning', ''))[:200]}")
    print(f"  cost/time     : ${j['_meta']['estimated_cost_usd']:.6f} / "
          f"{j['_meta']['seconds']}s")

print("\n" + "=" * 78)
print("SCORE DISTRIBUTION")
print("=" * 78)
for s in range(1, 6):
    n = sum(1 for x in scores if x == s)
    print(f"  {s}: {'#' * n}{'' if n else ' (none)'}")

print("\n" + "=" * 78)
print("SUMMARY")
print("=" * 78)
for k, v in aggregate.items():
    print(f"  {k:<28} {v}")

### Reading the output

Fill these in from your own run, then carry the numbers into `evaluation_memo.md`,
replacing the `[STEP 10]` markers.

Three things to look for, and what each would mean:

**Did the deterministic gate catch cases 4 and 5?** It should. Case 4 mentions SCHUFA
when the record says no bureau was used; case 5 states an income reason when the record
has no reason at all. If the gate missed either, the keyword lists in
`REASON_KEYWORDS` are too narrow, and that is a finding about brittle rule-based
checking worth writing up honestly.

**Did the judge score case 3 low?** That case is warm, condescending and promises a
better future outcome. If it scored 4 or 5, the judge is being seduced by friendly
register, which is exactly the same-family style bias predicted in the Step 4 bias
analysis. Report that as a limitation rather than tuning until it looks good.

**Did the judge stay in its lane?** Cases 4 and 5 have serious compliance faults but
acceptable tone. A judge that scored them low has drifted outside its instructions and
is double-counting faults the rules already caught, which would inflate any combined
metric you build on top.

## Limitations of this run

- Five cases, one language, one model version, one judge model. Not a sample.
- Letters are hand-written to contain specific faults, so they are not representative of
  real model output. This run tests the **evaluation**, not the drafting model.
- Single pass at temperature 0. Judge variance is unmeasured. Running each case five
  times and reporting the spread would be the first thing to add.
- Judge and drafting model share a training lineage, which biases the judge toward text
  that sounds machine-generated.
- The deterministic checks use English keyword matching. Production letters are German,
  and the keyword lists would need to be rebuilt with a native speaker.